# Native MolSysMT prepared-input consumption

Initial acceptance slice for DockingMT #33. Published Vina preparations are comparison
inputs, not chemical ground truth. This notebook uses public MolSysMT form conversion
and tree retrieval; raw PDBQT retains its unassessed preparation status. SDF dialect gaps
and chemical readiness are recorded in `native_molsysmt_formats.md`. No molecular source
is rewritten or prepared chemically.

In [1]:
import json
from pathlib import Path
import molsysmt as msm
import pyunitwizard as puw
import dockingmt as dmt
from molsysmt.form.file_pdbqt import get_torsion_tree

root = next(path for path in [Path.cwd(), *Path.cwd().parents]
            if (path / "tests/data/vina_torsions/1iep_receptor.pdbqt").is_file())
data = root / "tests/data/vina_torsions"
print(json.dumps({"MolSysMT_runtime_version": msm.__version__,
                  "qualified_provider_source": "eb0549b50689b2af5d8fbedb2687fb5746d43de6",
                  "Vina_reference_source": "3c65c0b3e6c2c1d183f6a175ecb65e3c5ba91645"}, indent=2))

{
  "MolSysMT_runtime_version": "0.22.4+118.g03b318549.dirty",
  "qualified_provider_source": "eb0549b50689b2af5d8fbedb2687fb5746d43de6",
  "Vina_reference_source": "3c65c0b3e6c2c1d183f6a175ecb65e3c5ba91645"
}


## Preserve supplied PDBQT assignments and the separate declared tree

The native projection explicitly discards the tree after retaining it. Native output
requires `typing_scheme='autodock4'`; this declaration does not assign types or charges.

In [2]:
rows = []
for name in ["1iep_ligand", "1s63_ligand", "5x72_ligand_p59", "5x72_ligand_p69"]:
    source = data / f"{name}.pdbqt"
    tree = get_torsion_tree(source)
    native, report = msm.convert(source, to_form="molsysmt.MolSys",
                                 discard_torsion_tree=True, return_report=True)
    payload = msm.convert(native, to_form="string:pdbqt_text",
                          typing_scheme="autodock4", torsion_tree=tree)
    rows.append({"input": name,
                 "atoms": msm.get(native, element="system", n_atoms=True),
                 "branches": len(tree["branch_atom_pairs"]),
                 "declared_TORSDOF": tree["torsdof"],
                 "conversion_outcome": report.outcome,
                 "output_form": msm.get_form(payload)})
print(json.dumps(rows, indent=2))

[
  {
    "input": "1iep_ligand",
    "atoms": 40,
    "branches": 7,
    "declared_TORSDOF": 7,
    "conversion_outcome": "lossy",
    "output_form": "string:pdbqt_text"
  },
  {
    "input": "1s63_ligand",
    "atoms": 30,
    "branches": 6,
    "declared_TORSDOF": 6,
    "conversion_outcome": "lossy",
    "output_form": "string:pdbqt_text"
  },
  {
    "input": "5x72_ligand_p59",
    "atoms": 25,
    "branches": 2,
    "declared_TORSDOF": 2,
    "conversion_outcome": "lossy",
    "output_form": "string:pdbqt_text"
  },
  {
    "input": "5x72_ligand_p69",
    "atoms": 25,
    "branches": 2,
    "declared_TORSDOF": 2,
    "conversion_outcome": "lossy",
    "output_form": "string:pdbqt_text"
  }
]


## Run the existing 1IEP docking path using explicit MolSysMT strings

File-to-string bridges preserve original bytes. DockingMT removes `pdbqt_text:` at
the external Vina boundary. This exploratory execution verifies consumption, capture
and internal consistency; it does not qualify binding affinity or chemical preparation.

In [3]:
receptor = msm.convert(data / "1iep_receptor.pdbqt", to_form="string:pdbqt_text")
partner = msm.convert(data / "1iep_ligand.pdbqt", to_form="string:pdbqt_text")
problem = dmt.DockingProblem(
    receptor=receptor, partner=partner,
    search_domain=dmt.BoxRegion(center=puw.quantity([15.190, 53.903, 16.917], "angstrom"),
                                size=puw.quantity([20, 20, 20], "angstrom")),
    metadata={"partner_state_id": "published-1iep-ligand", "receptor_state_id": "published-1iep-receptor"})
result = dmt.dock(problem, protocol=dmt.VinaProtocol(cpu=1, seed=42, n_poses=2,
                  exhaustiveness=1, capture_backend_inputs=True))
record = result.to_dict()
audit = dmt.audit_result(record)
assert audit["status"] == "consistent"
assert all(record["provenance"]["preparation"][role]["assessment"] == "unassessed"
           for role in ("receptor", "partner"))
print(json.dumps({"poses": len(result), "internal_consistency": audit["status"],
                  "preparation_assessment": {role: record["provenance"]["preparation"][role]["assessment"]
                  for role in ("receptor", "partner")},
                  "input_sha256": {role: record["provenance"]["backend_artifacts"][role]["sha256"]
                  for role in ("receptor", "partner")}}, indent=2))

{
  "poses": 1,
  "internal_consistency": "consistent",
  "preparation_assessment": {
    "receptor": "unassessed",
    "partner": "unassessed"
  },
  "input_sha256": {
    "receptor": "f13cf3b36f61d87c3b58983e0b8ecf1c3456a685eb86dfe9ccfb139c7bdc2586",
    "partner": "15fb35648d8c18c70317842f3a0631b73a19429c710a037ab07310084d579bb8"
  }
}
